#**PLEASE SAVE A COPY OF THIS NOTEBOOK TO SUBMIT**


# Modeling: MultiModal AI — Homework 3
**MAS.S60 / 6.S985 • Spring 2026 • MIT**

In this homework, you will explore Vision-Language Models (VLMs) and gain hands-on experience fine-tuning one.

---

## Environment Setup

Go to the top menu:  
Runtime → Change runtime type → Hardware accelerator → Choose "A100"

If you do not have Colab Pro, you can sign up for a free student Colab Pro account here:  
https://colab.research.google.com/signup


# Part 1: Reading & Reflection (20 points)

### Required Reading
[Multimodal Few-Shot Learning with Frozen Language Models](https://arxiv.org/pdf/2106.13884)

[Quality Not Quantity: On the Interaction between Datase Design and Robustness of CLIP
](https://arxiv.org/pdf/2208.05516.pdf)

[Generative AI: Here to stay, but for good?](https://www.sciencedirect.com/science/article/pii/S0160791X2300177X)

---

### Questions
1. What types of multimodal data noise are typically present in multimodal datasets, and how can they negatively impact the performance of a model during training? Can you provide examples of multimodal data points that might be considered noisy? Furthermore, how might we develop estimators capable of distinguishing between noisy and noise-free multimodal data pairs? If you have unlimited fundings to use for data filtering and data cleaning, what would be the ideal way to clean the multimodal dataset?

2. What is the intuition of utilizing frozen large language models as the backbone for multimodal tasks? Which types of encoders would facilitate the integration of diverse information into a format understandable by LLMs? How do these LLMs process and interpret information from different modalities?

3. Ensuring the effectiveness of multimodal foundation models through high-quality instruction tuning is vital. A study detailed at [here](https://arxiv.org/pdf/2402.04333.pdf) introduces a strategy for selecting significant data specifically suited for enhancing instruction tuning for language models. A primary challenge in this approach is determining which data are most crucial for targeted instruction tuning. How can we accurately identify and select the most impactful data for enhancing instruction tuning in multimodal foundation models? Given the complexity of diverse and multimodal information, what strategies can ensure the effectiveness of instruction tuning data for specific tasks?

4. With the advancement of generative AI, distinguishing between AI-generated and human-created content is becoming increasingly challenging. Besides watermarking, which has its limitations, are there other effective methods to differentiate between AI-generated and human-created content across various modalities (text, audio, video, image)? Or is it becoming virtually impossible to make this distinction?

5. For state-of-the-art video generation models like Sora, Yann Lecun mentioned in [here](https://twitter.com/ylecun/status/1758740106955952191) that Sora does not understand the real world and its corresponding physical rules. Do you agree with this view? Can the future development of generative AI systems truly incorporate real-world knowledge, or are they limited in this aspect? Is pursuing generative AI a viable path towards achieving Artificial General Intelligence (AGI)?


1: Multimodal datasets often suffer from poor alignment between modalities. This is especially true with web-crawled datasets, where the caption is oftentimes not a description of the contents in the picture, but some other form of description (such as the source of the image or a social media tagline). For example, my girlfriend often captions her pictures taken over spring break with "best week ever #springbreak" instead of "picture of a sunset at the beach".

This can actively degrade the quality of a dataset, in that a VLM trained on X good data points + Y bad data points will perform worse than a VLM trained on just the X good data points. A good way to distinguish a good data point from a bad data point is to use a well-performing model that you know was trained on good data to remove examples whose text and image embeddings have low cosine similarity in said model's token embedding space. However, the best solution (given infinite funds) will always be using a group of human experts to re-annotate your dataset to not only remove noisy data points but convert them to good data points.



2: LLMs are remarkably intelligent, with vast knowledge bases and a the ability to learn trends from a small number of examples. The idea for how to best turn a natively text-only LLM into a pseudo-LMM is to embed your input images into the same token embedding space which the LLM stores its text tokens, thus creating a "text representation" of the image. This then allows you to use the LLMs strengths, including its knowledge base and its intelligence, without having to train on the entire internet like frontier labs do when creating their LLMs. Additionally, for various known and unknown reasons, fine-tuning an LLM on a small multimodal dataset can actually decrease its performance.

The best types of encoders are obviously those which encode images in the same vector embeddings as their textual descriptions, because this allows the text-only LLMs to best understand them. The LLM can then process and store the image embedding as if it was a textual description.



3: Training on the data with subsets or even individual data points excluded will allow you to see which data points contribute most to the decrease in loss for a given objective. You can also use gradient similarity during training to achieve the same goal, as they did in the LESS paper. It is also important for multimodal tasks to have strong modal alignment, as discussed previously. As such, removing data points which have weak multimodal alignment is also probably a good first step.

This is because bad data (that is, data which is not well aligned and/or does not help reduce validation loss) is not only useless but often actively harmful, and is capable of poisoning the model's outputs and reducing their quality.



4: While specific LLMs have a certain "style" that can be learned through prolonged interaction with them just like a human, LLMs as a whole, especially open models with less censorship and shorter instruction cards, do not speak meaningfully differently from educated and intelligent native English speakers (hence the use of em-dashes). Thus, it is virtually impossible to tell if a piece of text was generated by a modern state of the art LLM with any high degree of certainty unless it has an artifact in it like "as a large language model...".

For images and videos it is a bit more feasible, as advanced pixel-by-pixel analysis can detect specific visual artifacts which are the result of diffusion models. While it is getting harder and harder for humans to detect if a specific image or video was AI generated, it will probably be possible for algorithms to do so for many years to come, although the computational cost may get higher and the certainty lower.



5: That was then, this is now. The very existence of Genie 3 as a technology proves Lecun wrong in this regard. Indeed, Yann appears to be eerily prophetic in his ability to consistently call things impossible only for them to be achieved within the next calendar year.

On a more serious note, you and I are the products of our multimodal training data, accumulated over our 20-something years of life. If we can understand physics, then there is no reason that an LMM trained on large enough amounts of multimodal data couldn't eventually understand physics just as well as us. Indeed, I expect that given enough time and data and compute there will eventually be a model whose physical intuition is superior to that of the average human.

# Part 2: Testing and Fine-tuning VLMs (100 points)

# Problem 1: GPU Verification and Library Installation

Run the following code cell to verify that your environment is correctly configured.

This step ensures that **PyTorch** and **CUDA** can access the GPU.  
When the setup is correct, a **secret word** will appear in the output.

---

### In Your PDF Submission

Include:
- A **screenshot** or **code snippet** showing the printed GPU information.  
- The **secret word** displayed by your verification cell.

---

In [1]:
!pip install transformers accelerate bitsandbytes pillow torch -q

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t = torch.randn(2, 3, device=device)
KEY = 42
cipher_bytes = [99, 10, 102, 101, 124, 111, 10, 103, 103, 107, 99]

if t.is_cuda:
    cipher = torch.tensor(cipher_bytes, dtype=torch.uint8, device=device)
    decoded = torch.bitwise_xor(cipher, KEY)
    torch.cuda.synchronize()
    secret = bytes(decoded.tolist()).decode("ascii")
    print("SECRET_WORD:", secret)
else:
    print("SECRET_WORD: (not on GPU)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device count: 1
GPU name: NVIDIA A100-SXM4-40GB
SECRET_WORD: I LOVE MMAI


The word is I LOVE MMAI, and I have included a screenshot in my latex pdf submission.

# Problem 2: Prepare Your Dataset (20 points)

## **PLEASE READ THIS ENTIRE SECTION BEFORE PROCEDING**

For Problem 2, you will **use the dataset you have collected from Homework 1 and Homework 2 or a completely new one if you prefer** to fine-tune a Vision-Language Model (VLM).

Even if your original data isn't image-based (e.g. it's audio, time-series, or text), you should find a way to **visualize it** meaningfully. The dataset you prepare will serve as the foundation for model fine-tuning in later steps.

---

### How to Convert Your Project Data Into Images

**If your project is not originally image-based, consider these ideas to generate visual input:**

| Data Type                    | Visual Representation Example                          |
|-----------------------------|---------------------------------------------------------|
| Time-series / sensor data   | Line plots or multi-panel charts (with axis labels)     |
| Audio / Music / Physiology  | Spectrograms or waveform plots                         |
| 3d data (point clouds, CAD) | Rendering/splicing into 2D images

You are encouraged to be **creative and domain-specific** in your visualizations.

**You will need to explore ways to convert your data into images if it does not already consist of this modality. Research on your own and come up with the needed code to do so. If you are still stuck on figuring this out, please reach out to a TA for help!**

### Download Example Training Data

The next block of code will download an example dataset and create a folder named `mmai-data/`.  
Inside this folder, you will find:

```
mmai-data/
├── images/
│   ├── 1.jpg
│   └── 2.jpg
└── data.jsonl
```

The file `data.jsonl` contains your training annotations.  
Each line represents one training example with the following fields:

```json
{
  "image": "images/1.jpg",
  "question": "List objects you see.",
  "answer": "cat, sofa, blanket, remote, cushion"
}
```

---

### Your Task

Now, prepare your own dataset following the same structure as the example.


Example structure:

```
mmai-data/
├── images/
│   ├── image_01.jpg
│   ├── image_02.jpg
│   ├── ...
└── data.jsonl
```

As part of this task. You should split the data into a train and test split. **The test split should consist of the images of data that you will not use in training.**


In [10]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!cp /content/drive/MyDrive/mmai-train.zip /content/
!unzip /content/mmai-train.zip | tail -5

Mounted at /content/drive
replace mmai-train/data.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace mmai-train/images/v_00668_13.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace mmai-train/images/v_00395_9.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace mmai-train/images/v_00284_4.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: mmai-train/images/v_00340_8.jpg  
  inflating: mmai-train/images/v_00284_9.jpg  
  inflating: mmai-train/images/v_00395_4.jpg  
  inflating: mmai-train/images/v_00089_1.jpg  
  inflating: mmai-train/images/v_00458_1.jpg  


In [11]:
import os, shutil, zipfile
from pathlib import Path

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
# Upload your dataset as a zip file to Google Drive, then
# replace the URL below with your own Google Drive share link.
#
# Your zip should unpack into a folder called mmai-data/ with:
#   mmai-data/
#   ├── images/
#   │   ├── image_01.jpg
#   │   ├── ...
#   │   └── image_16.jpg
#   └── data.jsonl
#
# Each line in data.jsonl should be a JSON object with three
# fields: "image", "question", and "answer". For example:
#
#   {"image": "images/1.jpg", "question": "List objects you see.", "answer": "cat, sofa, blanket, remote, cushion, tail, paw"}
#   {"image": "images/2.jpg", "question": "List objects you see.", "answer": "car, truck, road, bridge, exit sign, lamppost, building, sky"}
#
URL = "https://drive.google.com/file/d/14S_HZA4sbCCZnMsHS1o_jmK7rWEGwDiw/view?usp=sharing"
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

DATA_DIR = Path("/content/")
DATA_DIR.mkdir(parents=True, exist_ok=True)

!pip -q install gdown
import gdown

print("Downloading…")
downloaded_path = gdown.download(URL, output=None, quiet=False, fuzzy=True)
if not downloaded_path or not os.path.exists(downloaded_path):
    raise RuntimeError("Download failed. Check the URL or your Drive permissions.")

src = Path(downloaded_path)
dst = DATA_DIR / src.name
if src.resolve() != dst.resolve():
    shutil.move(str(src), str(dst))

print(f"\nFile saved to: {dst}")

if zipfile.is_zipfile(dst):
    extract_dir = DATA_DIR / dst.stem
    extract_dir.mkdir(exist_ok=True)
    print(f"Unzipping into: {extract_dir}")
    with zipfile.ZipFile(dst, "r") as zf:
        zf.extractall(extract_dir)
    print("Unzip complete.")

if dst.suffix.lower() == ".jsonl":
    print("\nSet this in your training cell:")
    print(f'DATA_JSONL = "{dst}"')

Downloading…


Downloading...
From: https://drive.google.com/uc?id=14S_HZA4sbCCZnMsHS1o_jmK7rWEGwDiw
To: /content/mmai-data.zip
100%|██████████| 346k/346k [00:00<00:00, 114MB/s]


File saved to: /content/mmai-data.zip
Unzipping into: /content/mmai-data
Unzip complete.


In [12]:
import json, os, shutil

# Create the mmai-data folder
os.makedirs("mmai-data/images", exist_ok=True)

# Copy images
!cp mmai-train/images/* mmai-data/images/

# These are our 4 held-out test examples
holdout_ids = ["v_00000_4", "v_00001_3", "v_00002_3", "v_00013_1"]

# Read all entries from our JSONL
with open("mmai-train/data.jsonl") as f:
    all_entries = [json.loads(line) for line in f]

# Split into train and holdout
train_entries = [e for e in all_entries if not any(h in e["image"] for h in holdout_ids)]
holdout_entries = [e for e in all_entries if any(h in e["image"] for h in holdout_ids)]

# Write training data
with open("mmai-data/data.jsonl", "w") as f:
    for e in train_entries:
        f.write(json.dumps(e) + "\n")

# Save holdout for later
with open("holdout.jsonl", "w") as f:
    for e in holdout_entries:
        f.write(json.dumps(e) + "\n")

print(f"Training examples: {len(train_entries)}")
print(f"Held-out test examples: {len(holdout_entries)}")
for e in holdout_entries:
    print(f"  {e['image']} | Q: {e['question']} | A: {e['answer']}")

Training examples: 2736
Held-out test examples: 4
  images/v_00000_4.jpg | Q: Which direction will I need to move to exit the vehicle? | A: Left.
  images/v_00001_3.jpg | Q: Are there any obstacles on the sidewalk ahead? | A: On your right there are several battery carts.
  images/v_00002_3.jpg | Q: Is it safe to cross the street now? | A: Yes.
  images/v_00013_1.jpg | Q: Is the cashier helping me with the payment? | A: Yes.


## Questions to Answer:

*   Explain some possible issues with converting non-image data into images (even if you did not have to do so, discuss what could be some issues).

*   What are some possible issues with using visual representations of your data. Discuss some drawbacks of doing this (if you did not have to do the conversion as your data was already in the form of images, then discuss the drawbacks of converting those images to another modality like text, audio, etc.).

* Discuss the strategy you decided on how to split your data into train/test splits. Why did you settle on this? Were any other alternative splits considered?



One problem is that some data just doesn't have a good visual representation. For example, sound data can be expressed visually as a waveform, but it is very hard to tell what exactly is being said just from looking at the waveform. Even worse is anything that is strongly temporal, like time series data, since images don't really encode temporal relationships very well.

My data is videos with questions posed by blind people referencing the videos. My strategy was to sample the exact frame at which a question was asked and use that as the image representing the data point. The problem here is that a large percentage of the questions are asked about things that had just happened previously, so oftentimes the event or object in question was no longer in frame by the time the question was asked.

The original dataset's videos and questions (which we later modified through the addition of several other fields) were already divided into train and test splits, so we just used their preexisting splits.

# Problem 3: Baseline Inference (10 points)

# Problem 3.1 Load the Model

Begin by running the following code to **load the base model** into memory. This step is required before training or making predictions.


In [13]:
import io, requests, torch
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

# 1) Load model + processor (processor handles BOTH text + vision)
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
print("Model and tokenizer loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model and tokenizer loaded successfully.


# Problem 3.2: Run the Model on Your 4 Held-Out Images

In this step, you will use the **pre-trained** `Qwen2.5-VL-3B-Instruct` model (no fine-tuning yet) to answer questions about the **held-out images** that were **not used in training**. You will then compare the model’s predictions with the ground-truth labels and reflect on its performance.

---

## Instructions

1. **Select four held-out images**  
   Choose four test images from your dataset that were excluded from training and prompt development.

2. **Ask a consistent question**  
   Use the same question for all images, or a small set of label-aligned questions.

3. **Run the model**  
   Use the provided code cell to run inference with the pre-trained model.  

4. **Record your results**  
   For each image, collect the model’s raw output and compare it to the ground-truth label(s). If there are too many images, then show a few examples.

---

## Reflection (5–8 sentences)

After running the model on your four images, briefly discuss:
- **What worked?**  
  Which prompts or parameter settings produced better results?
- **What failed?**  
  Were there recurring failure modes (e.g., hallucinations, vague answers)?
- **Patterns in mistakes**  
  Did errors correlate with certain categories, lighting conditions, or question phrasing?

---

## Suggested Output Format

| Image ID/URL | Question | Model Output | Ground Truth | Result |
|---------------|-----------|---------------|---------------|---------|
| `img_001.jpg` | “What objects are visible?” | cat, sofa | cat, sofa | Correct |
| `img_002.jpg` | “What objects are visible?” | road, truck, sign | road, car, sign | Incorrect |

In [14]:
import io
import os
import requests
import torch
from typing import Optional, Dict, Any
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration


# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
IMAGE_URL: str = "http://images.cocodataset.org/val2017/000000039769.jpg"
QUESTION: str = "How does the image make you feel? Show a list of emotion labels only."
SYSTEM_PROMPT: str = "You are a helpful assistant."
MAX_NEW_TOKENS: int = 128
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================


# SYSTEM CONFIG
DO_SAMPLE: bool = False           # set True for non-greedy decoding
TEMPERATURE: float = 0.7          # used only if DO_SAMPLE=True
TOP_P: float = 0.9                # used only if DO_SAMPLE=True
MODEL_ID: str = "Qwen/Qwen2.5-VL-3B-Instruct"
FORCE_CPU: bool = False           # force CPU even if CUDA is available
DTYPE_IF_GPU = torch.bfloat16     # prefer bfloat16 on recent GPUs/Colab
DTYPE_IF_CPU = torch.float32

def get_device_and_dtype() -> tuple[torch.device, torch.dtype, Optional[Dict[str, Any]]]:
    """Choose device/dtype and (optionally) a device_map for accelerate-style placement."""
    use_cuda = torch.cuda.is_available() and not FORCE_CPU
    device = torch.device("cuda") if use_cuda else torch.device("cpu")
    torch_dtype = DTYPE_IF_GPU if use_cuda else DTYPE_IF_CPU
    device_map = "auto" if use_cuda else None
    return device, torch_dtype, device_map


def load_image_from_url(url: str) -> Image.Image:
    """Fetch image from URL and return a RGB PIL.Image with robust fallback."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    try:
        return Image.open(io.BytesIO(resp.content)).convert("RGB")
    except UnidentifiedImageError:
        # Fallback: write to disk then reopen (sometimes fixes truncated headers)
        tmp_path = "temp_image.jpg"
        with open(tmp_path, "wb") as f:
            f.write(resp.content)
        img = Image.open(tmp_path).convert("RGB")
        try:
            os.remove(tmp_path)
        except Exception:
            pass
        return img


def build_chat_messages(image: Image.Image, question: str) -> list[dict]:
    """Create a single-turn, image+text chat for Qwen-VL processors."""
    return [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]


def main() -> None:
    device, torch_dtype, device_map = get_device_and_dtype()

    # 1) Load model + processor
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map=device_map,
    )
    print("Model and processor loaded successfully.")

    # 2) Load image
    image = load_image_from_url(IMAGE_URL)

    # 3) Build chat
    messages = build_chat_messages(image, QUESTION)

    # 4) Apply chat template and preprocess
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")

    # 5) Move to the right device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 6) Generate
    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS)
    if DO_SAMPLE:
        gen_kwargs.update(dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P))

    with torch.no_grad():
        gen_ids = model.generate(**inputs, **gen_kwargs)

    # 7) Decode
    out = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    print("\n=== MODEL OUTPUT ===")
    print(out)


if __name__ == "__main__":
    main()


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model and processor loaded successfully.

=== MODEL OUTPUT ===
system
You are a helpful assistant.
user
How does the image make you feel? Show a list of emotion labels only.
assistant
calm, cozy, relaxed, content, peaceful


In [15]:
SYSTEM_PROMPT = "You are a helpful assistant."
MAX_NEW_TOKENS = 128

with open("holdout.jsonl") as f:
    holdout = [json.loads(line) for line in f]

for ex in holdout:
    image = Image.open(os.path.join("mmai-data", ex["image"])).convert("RGB")

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": ex["question"]},
        ]},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    output = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

    print(f"Image: {ex['image']}")
    print(f"Question: {ex['question']}")
    print(f"Model Output: {output}")
    print(f"Ground Truth: {ex['answer']}")
    print("---")

Image: images/v_00000_4.jpg
Question: Which direction will I need to move to exit the vehicle?
Model Output: system
You are a helpful assistant.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered through the driver's side door, you should move towards the passenger side door to exit. If you entered through the passenger side door, you should move towards the driver's side door to exit.
Ground Truth: Left.
---
Image: images/v_00001_3.jpg
Question: Are there any obstacles on the sidewalk ahead?
Model Output: system
You are a helpful assistant.
user
Are there any obstacles on the sidewalk ahead?
assistant
Yes, there are several obstacles on the sidewalk ahead. There are multiple scooters parked along the sidewalk, and some of them appear to be in motion or moving quickly. Additionally, there is a tree trunk visible on the left side of the image, which could also pose an

The model is being overly safe to the point of actually being useless and unsafe. My dataset involves questions like "Is it safe to cross the street now?" and you can clearly see the model give a long, fluff-filled answers that include irrelevant information, don't actually answer the question, and say things like "Whether it is safe to cross the street depends on...".

Furthermore, the model is worried about the wrong things. When asked if there are any obstacles ahead, instead of pointing out the carts immediately ahead and to the right that the person may trip on, it warns that there is a tree off the path which may be potentially hazardous if it were to fall as the person walks by. This is obviously a very niche risk, and much less important than the immediate risk of large heavy battery carts that the human may walk into.

When asked which direction the human should exit the car, the model gives a completely useless and nonsensical answer that is potentially very dangerous, saying that the user should exit the vehicle using whichever door they did not use to enter. Given that passengers of taxi cabs usually enter and exit on the right side, since there is traffic on the left, this is obviously problematic.

At least the model got the cashier question right, although it could have answered in a single word instead of a paragraph.

# Problem 4: Prompt Engineering (15 points)

In this step, you'll experiment with **prompt design** to explore how different instructions influence model performance.

---

### Instructions

1. Modify the **`SYSTEM_PROMPT`** variable inside the **CHANGE ME** section of the code above.  
2. Re-run the corresponding code cell to observe how the model's responses change.  
3. Test various prompt strategies, such as:
   - Adding **examples** (few-shot prompting)
   - Restricting **answer formats** (e.g., "Answer with one word")
   - Asking for **explanations** or **step-by-step reasoning**
4. Compare your new results with the baseline output.

---

### Reflection

In your write-up, discuss:
- Which types of prompt changes improved performance?  
- Did adding context or structure help the model reason more effectively?  
- Were there any surprising or inconsistent results?


In [16]:
prompts_to_try = [
    "You are a helpful assistant.",
    "You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.",
    "You are a visual assistant for blind users. Answer in one sentence. Focus on safety and obstacles.",
]

with open("holdout.jsonl") as f:
    holdout = [json.loads(line) for line in f]

for prompt in prompts_to_try:
    print(f"\n{'='*60}")
    print(f"SYSTEM PROMPT: {prompt}")
    print(f"{'='*60}")

    for ex in holdout:
        image = Image.open(os.path.join("mmai-data", ex["image"])).convert("RGB")

        messages = [
            {"role": "system", "content": [{"type": "text", "text": prompt}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": ex["question"]},
            ]},
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=128)

        output = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

        print(f"  Q: {ex['question']}")
        print(f"  Model: {output}")
        print(f"  Truth: {ex['answer']}")
        print()


SYSTEM PROMPT: You are a helpful assistant.
  Q: Which direction will I need to move to exit the vehicle?
  Model: system
You are a helpful assistant.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered through the driver's side door, you should move towards the passenger side door to exit. If you entered through the passenger side door, you should move towards the driver's side door to exit.
  Truth: Left.

  Q: Are there any obstacles on the sidewalk ahead?
  Model: system
You are a helpful assistant.
user
Are there any obstacles on the sidewalk ahead?
assistant
Yes, there are several obstacles on the sidewalk ahead. There are multiple scooters parked along the sidewalk, and some of them appear to be in motion or moving quickly. Additionally, there is a tree branch hanging over the sidewalk, which could pose a potential hazard if it were to fall. The presence of th

While it remains confidently incorrect about many things, the model does become substantially more useful when told that it is an assistant to blind users. It begins to give spatial/directional information detailing the locations of objects instead of just acknowledging their presence.

When additionally told to answer in single sentences focusing on safety and obstacle, the model becomes even more useful, removing most of the useless fluff in its answers and instead providing short, actionable, and (generally) useful answers that actually answer the question instead of diverging into irrelevant exposition.

# Problem 5: LoRA Fine-Tuning (20 points)

In this step, you'll fine-tune a **Vision-Language Model (VLM)** using **LoRA (Low-Rank Adaptation)** on your dataset.  
This exercise will help you understand how different hyperparameters influence performance, GPU memory usage, and output quality.

### Instructions

Run the code block below.  
If you followed the **`mmai-data`** example, the script should automatically detect and load your training dataset.

### Adjust and experiment with

- **Number of epochs** (`NUM_EPOCHS`)
- **Learning rate** (`LR`)
- **Batch size per device** (`BSZ_PER_DEV`)
- **Gradient accumulation steps** (`GRAD_ACCUM`)
- **Evaluation split ratio** (`EVAL_SPLIT`)
- **Random seed** (`SEED`)
- **Sequence length** (`MAX_SEQ_LEN`)
- **Image resolution** (`SHORTEST_EDGE`)
- **LoRA rank** (`LORA_R`)
- **LoRA alpha** (`LORA_ALPHA`)
- **LoRA dropout** (`LORA_DROPOUT`)
- **LoRA target modules** (`LORA_TARGET`)

---

```python
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

# (Modify the parameters below in the Colab cell)

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================
```

### Q&A

**Q:** What should I do if I encounter an out-of-memory issue?  
**A:** Your image might be too large. Try resizing it by adding the following line back into your code and experiment with different pixel values:

```python
img.thumbnail((128, 128))  # NOTE: If you run into an out-of-memory error, try adding this line back.
```


In [20]:
import json, random
random.seed(42)
with open("/content/mmai-data/data.jsonl") as f:
    lines = f.readlines()
lines = random.sample(lines, 500)
with open("/content/mmai-data/data.jsonl", "w") as f:
    f.writelines(lines)
print(f"Subsampled to {len(lines)} examples")

Subsampled to 500 examples


In [28]:
# ==== Qwen2.5-VL-3B-Instruct • FP16 LoRA ====

from IPython.display import display, HTML
import os, io, json, requests, torch, random, hashlib
from dataclasses import dataclass
from typing import Any, Dict, List
from PIL import Image
from torch.utils.data import Dataset
import torch.nn as nn
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model

# Environment hygiene
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
# Training hyperparameters
NUM_EPOCHS: int  = 1
LR: float        = 1e-4
BSZ_PER_DEV: int = 1
GRAD_ACCUM: int  = 8
EVAL_SPLIT: float = 0.1
SEED: int        = 42

# Collator / sequence shaping
MAX_SEQ_LEN: int = 512   # try 384 if VRAM is tight

# Image preprocessing
SHORTEST_EDGE: int = 288  # smaller saves VRAM
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================


# SYSTEM CONFIG
# Paths
DATA_JSONL: str  = "/content/mmai-data/data.jsonl"
OUTPUT_DIR: str  = "/content/qwen2_5_vl_lora_fp16_t4"

MODEL_ID: str     = "Qwen/Qwen2.5-VL-3B-Instruct"
CACHE_DIR: str    = "/content/cache_images"
IMAGE_TIMEOUT: int = 15

# LoRA configuration (attention-only keeps memory low)
LORA_R: int          = 4
LORA_ALPHA: int      = 8
LORA_DROPOUT: float  = 0.05
LORA_TARGET: list[str] = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Device / dtype policy
FORCE_CPU: bool   = False
DTYPE_IF_GPU      = torch.float16
DTYPE_IF_CPU      = torch.float32


# Repro and cache dirs
torch.manual_seed(SEED); random.seed(SEED)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# --------------------
# Demo data (create if missing)
# --------------------
def _ensure_sample_data(path: str):
    # Ensure parent directory exists
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if os.path.exists(path):
      display(HTML(
          "<div style='color:white; background-color:#2e7d32; padding:10px; border-radius:6px;'>"
          "<strong>Using custom training data:</strong> "
          f"Loaded dataset from <code>{path}</code>. "
          "Proceeding with user-provided images and JSONL file."
          "</div>"
      ))
      return

    demo = [
        {
            "image": "http://images.cocodataset.org/val2017/000000039769.jpg",
            "question": "List objects you see.",
            "answer": "cat, sofa, blanket, remote, cushion, tail, paw"
        },
        {
            "image": "http://images.cocodataset.org/val2017/000000001532.jpg",
            "question": "List objects you see.",
            "answer": "car, truck, road, bridge, exit sign, lamppost, building, sky"
        },
    ]
    with open(path, "w") as f:
        for r in demo: f.write(json.dumps(r) + "\n")

    # Print a red warning box (works in Colab/Jupyter)
    display(HTML(
        "<div style='color:white; background-color:#b71c1c; padding:10px; border-radius:6px;'>"
        "<strong>Warning:</strong> No dataset found — using built-in <code>sample data</code> (2 demo images). "
        "Please replace with your own dataset of at least 20 images for training."
        "</div>"
    ))

_ensure_sample_data(DATA_JSONL)


# --------------------
# Minimal JSONL dataset
# --------------------
class JsonlVisionLangDataset(Dataset):
    def __init__(self, jsonl_path: str):
        self.samples: list[dict] = []
        with open(jsonl_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                ex = json.loads(line)
                if {"image","question","answer"} - set(ex.keys()): continue
                self.samples.append(ex)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx: int) -> Dict[str, Any]: return self.samples[idx]

full_ds = JsonlVisionLangDataset(DATA_JSONL)

# Manual split
n = len(full_ds); n_val = max(1, int(n * EVAL_SPLIT))
idx = list(range(n)); random.shuffle(idx)
val_idx = set(idx[:n_val])
train_data = [full_ds[i] for i in range(n) if i not in val_idx]
val_data   = [full_ds[i] for i in range(n) if i in val_idx]

class ListDataset(Dataset):
    def __init__(self, data_list): self.data_list = data_list
    def __len__(self): return len(self.data_list)
    def __getitem__(self, i): return self.data_list[i]


# --------------------
# Cache images locally (avoid network hiccups)
# --------------------
BASE_DIR = os.path.dirname(DATA_JSONL)

def cache_image(url_or_path: str) -> str:
    # Remote URL: download and cache
    if url_or_path.startswith(("http://", "https://")):
        h = hashlib.md5(url_or_path.encode()).hexdigest()
        local = os.path.join(CACHE_DIR, f"{h}.jpg")
        if not os.path.exists(local):
            r = requests.get(url_or_path, timeout=IMAGE_TIMEOUT); r.raise_for_status()
            with open(local, "wb") as f: f.write(r.content)
        return local

    # Local path: make absolute relative to the JSONL file
    candidate = url_or_path
    if not os.path.isabs(candidate):
        candidate = os.path.join(BASE_DIR, url_or_path)

    if not os.path.exists(candidate):
        raise FileNotFoundError(
            f"Image not found: {candidate} (from '{url_or_path}'). "
            f"Expected under {BASE_DIR}/"
        )
    return candidate


for ex in train_data: ex["image"] = cache_image(ex["image"])
for ex in val_data:   ex["image"] = cache_image(ex["image"])

train_ds = ListDataset(train_data)
val_ds   = ListDataset(val_data)


# --------------------
# Image loader
# --------------------
def load_image(img_path: str) -> Image.Image:
    img = Image.open(img_path).convert("RGB")
    # img.thumbnail((128, 128))  # NOTE: if you run into out of memory error, try adding this line back
    return img


# --------------------
# Processor + Model (FP16 on GPU, FP32 on CPU)
# --------------------
use_cuda = torch.cuda.is_available() and not FORCE_CPU
torch_dtype = DTYPE_IF_GPU if use_cuda else DTYPE_IF_CPU
device_map = "auto" if use_cuda else None

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,            # transformers v5 uses 'dtype'
    device_map=device_map,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Smaller images to save VRAM
try:
    if hasattr(processor, "image_processor") and hasattr(processor.image_processor, "size"):
        processor.image_processor.size = {
            "shortest_edge": int(SHORTEST_EDGE),
            "longest_edge": int(SHORTEST_EDGE * 4),
        }
        print(f"Set image shortest_edge to {SHORTEST_EDGE}, longest_edge to {SHORTEST_EDGE * 4}")
except Exception as e:
    print("Skip image size tweak:", e)

# Enable gradient checkpointing; avoid k-bit prep (saves VRAM)
#model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False


# --------------------
# LoRA (attention-only)
# --------------------
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


# --------------------
# Collator (truncate to keep sequences small)
# --------------------
@dataclass
class VLDataCollator:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        images, texts = [], []
        for ex in features:
            img = load_image(ex["image"])
            messages = [
                {"role": "user", "content": [
                    {"type":"image","image": img},
                    {"type":"text","text": ex["question"]},
                ]},
                {"role": "assistant", "content": [
                    {"type":"text","text": ex["answer"]},
                ]},
            ]
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            images.append(img); texts.append(text)

        batch = self.processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels

        for im in images:
            try: im.close()
            except: pass

        return batch

collator = VLDataCollator(processor)


# --------------------
# FP16 loss trainer to avoid fp32 upcast OOM
# --------------------
class FP16CLMTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,   # v5 may pass this
        **kwargs,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # keep fp16 path if available

        # Shift for causal LM
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


# --------------------
# TrainingArguments (Transformers v5+ naming)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BSZ_PER_DEV,
    per_device_eval_batch_size=1,
    dataloader_num_workers=0,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=1,

    eval_strategy="no",              # keep simple; eval loop optional
    save_strategy="steps",
    save_steps=10_000,

    fp16=use_cuda, bf16=False,       # FP16 only if GPU
    gradient_checkpointing=True,
    optim="adamw_torch",
    report_to=[],
    remove_unused_columns=False,
)

trainer = FP16CLMTrainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

# Save LoRA adapters + processor
trainer.model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Training complete. LoRA adapters saved to:", OUTPUT_DIR)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Set image shortest_edge to 288, longest_edge to 1152


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 1,843,200 || all params: 3,756,466,176 || trainable%: 0.0491


Step,Training Loss
1,7.945479
2,7.637000
3,8.274776
4,7.904330
5,7.543725
6,7.405631
7,7.496619
8,7.000050
9,7.174064
10,6.462759


Training complete. LoRA adapters saved to: /content/qwen2_5_vl_lora_fp16_t4


In [29]:
!cp -r /content/qwen2_5_vl_lora_fp16_t4 /content/drive/MyDrive/lora_run5
print("Saved Run 5!")

Saved Run 5!


# **Questions to answer:**

1. Report the settings you used to get the best model.
  
2. Which hyperparameters did you find have the most impact in the model’s performance?

3. Why do you think that is?



I varied 4 things:

Number of training examples (because with my full dataset it takes over 30 minutes per epoch), number of epochs, learning rate, and gradient accumulation.

If you go down to the end of section 6 you can see me evaluate the models against the validation set.

The best model was Run 2: 450 examples, lr=1e-4, 1 epoch, ga=1, achieving a holdout loss of 2.8063

The pretrained baseline performs the worst, which is to be expected.

The training run with my full dataset actually also performed worse than my run with a smaller set, which was surprising. I can't really explain why this is the case no matter how much I try. Perhaps it is because I changed the base model too much with my thousands of fine-tuning examples.

Increasing epochs also actually made the model slightly worse, likely due to overfitting.

Decreasing learning rate also made the model slightly worse.

The parameter with the largest effect was gradient accumulation. Increasing it from 1 to 8 resulted in a substantial drop in model performance, larger than any other hyperparameter or change in training data size.

# Problem 6: Post-Training Evaluation (30 points)

# Problem 6.1 Load the Trained LoRA Adapter

Once your fine-tuning is complete, load the trained **LoRA adapters** back onto the original model to perform inference, that is, to generate predictions or analyze new images.

Simply run the code in the next code block.  
It will automatically attach your fine-tuned LoRA weights and prepare the model for evaluation.


In [30]:

# --------------------
# Inference with adapters
# --------------------
from peft import PeftModel
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    # device_map="cuda:0",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
ft_model.eval()
print("LoRA adapters loaded. Ready for inference.")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

LoRA adapters loaded. Ready for inference.


# Problem 6.2 Re-Test on Held-Out Images

Re-test the same **held-out images** used in your baseline evaluation.

Compare the **pre-trained** (in Step 2.2) and **fine-tuned** model outputs:

- Which questions showed improvement?  
- Did LoRA fine-tuning correct any earlier mistakes?  
- Were any new errors or biases introduced after fine-tuning?

Document your observations and include examples where possible.


In [33]:
import json, os, torch
from PIL import Image
from peft import PeftModel

SYSTEM_PROMPT = "You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers."
MAX_NEW_TOKENS = 128

with open("holdout.jsonl") as f:
    holdout = [json.loads(line) for line in f]

# --- BASELINE ---
print("="*60)
print("BASELINE: Pre-trained model (no fine-tuning)")
print("="*60)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
base_model.eval()

baseline_outputs = {}
for ex in holdout:
    image = Image.open(os.path.join("mmai-data", ex["image"])).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": ex["question"]},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        gen_ids = base_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    output = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    baseline_outputs[ex["image"]] = output
    print(f"  Q: {ex['question']}")
    print(f"  Baseline: {output}")
    print(f"  Truth: {ex['answer']}")
    print()

del base_model
torch.cuda.empty_cache()

# --- FINE-TUNED RUNS ---
run_dirs = [
    ("/content/drive/MyDrive/lora_run1", "Run1: 2742ex, lr=1e-4, 1ep"),
    ("/content/drive/MyDrive/lora_run2", "Run2: 450ex, lr=1e-4, 1ep"),
    ("/content/drive/MyDrive/lora_run3", "Run3: 450ex, lr=1e-4, 2ep"),
    ("/content/drive/MyDrive/lora_run4", "Run4: 450ex, lr=2e-5, 1ep"),
    ("/content/drive/MyDrive/lora_run5", "Run5: 450ex, lr=1e-4, ga=8"),
]

for run_path, run_label in run_dirs:
    print(f"\n{'='*60}")
    print(f"FINE-TUNED: {run_label}")
    print(f"{'='*60}")

    if not os.path.exists(run_path):
        print(f"  SKIPPED — not found: {run_path}")
        continue

    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    ft_model = PeftModel.from_pretrained(base, run_path)
    ft_model.eval()

    for ex in holdout:
        image = Image.open(os.path.join("mmai-data", ex["image"])).convert("RGB")
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": ex["question"]},
            ]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(ft_model.device)
        with torch.no_grad():
            gen_ids = ft_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
        output = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

        print(f"  Q: {ex['question']}")
        print(f"  Baseline:   {baseline_outputs[ex['image']]}")
        print(f"  Fine-tuned: {output}")
        print(f"  Truth:      {ex['answer']}")
        print()

    del ft_model, base
    torch.cuda.empty_cache()

print("\nDone! Compare baseline vs fine-tuned outputs above.")

BASELINE: Pre-trained model (no fine-tuning)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

  Q: Which direction will I need to move to exit the vehicle?
  Baseline: system
You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered from the front door, you would exit through the back door. If you entered from the back door, you would exit through the front door.
  Truth: Left.

  Q: Are there any obstacles on the sidewalk ahead?
  Baseline: system
You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.
user
Are there any obstacles on the sidewalk ahead?
assistant
Yes, there is an obstacle on the sidewalk ahead. It appears to be a large object or structure that may require caution when passing by. Please proceed with care and be aware of your surroundings.
  Truth: On your right there are several batter

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

  Q: Which direction will I need to move to exit the vehicle?
  Baseline:   system
You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered from the front door, you would exit through the back door. If you entered from the back door, you would exit through the front door.
  Fine-tuned: system
You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered from the right side, you should move to the left to exit.
  Truth:      Left.

  Q: Are there any obstacles on the sidewalk ahead?
  Baseline:   system
You are a visual assistant helping a visually

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

  Q: Which direction will I need to move to exit the vehicle?
  Baseline:   system
You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered from the front door, you would exit through the back door. If you entered from the back door, you would exit through the front door.
  Fine-tuned: system
You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers.
user
Which direction will I need to move to exit the vehicle?
assistant
To exit the vehicle, you should move in the direction opposite to where you entered. If you entered from the front, you'll need to move towards the back of the vehicle. If you entered from the side, you'll need to move towards the opposite side. Always check for traffic and pedestrians before ste

All fine-tuned models show substantial improvement over the baseline on most questions. Instead of providing long, hedgy, fluff-filled answers, they provide actually useful information.

However, things are not perfect. All models still fail to correctly say "left" when asked which direction to exit the car, even if most of them have superior performance on the other three questions. As such, directional understanding remains challenging for the models.

Additionally they are sometimes confidently incorrect, saying "No" when asked about the presence of obstacles where the base model at least listed some potential obstacles.

In [34]:
import json, os, torch
from PIL import Image
from peft import PeftModel
import torch.nn as nn

SYSTEM_PROMPT = "You are a visual assistant helping a visually impaired person navigate. Give short, clear, actionable answers."

with open("holdout.jsonl") as f:
    holdout = [json.loads(line) for line in f]

def compute_loss(model_to_eval, examples):
    total_loss = 0
    for ex in examples:
        image = Image.open(os.path.join("mmai-data", ex["image"])).convert("RGB")
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": ex["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": ex["answer"]},
            ]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(model_to_eval.device)
        labels = inputs["input_ids"].clone()
        labels[inputs["attention_mask"] == 0] = -100
        with torch.no_grad():
            outputs = model_to_eval(**inputs, labels=labels)
            total_loss += outputs.loss.item()
    return total_loss / len(examples)

# Baseline
print("Computing losses...\n")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto", low_cpu_mem_usage=True, trust_remote_code=True,
)
base_model.eval()
baseline_loss = compute_loss(base_model, holdout)
print(f"{'Baseline':<45} Loss: {baseline_loss:.4f}")
del base_model
torch.cuda.empty_cache()

# Each fine-tuned run
run_dirs = [
    ("/content/drive/MyDrive/lora_run1", "Run1: 2742ex, lr=1e-4, 1ep"),
    ("/content/drive/MyDrive/lora_run2", "Run2: 450ex, lr=1e-4, 1ep"),
    ("/content/drive/MyDrive/lora_run3", "Run3: 450ex, lr=1e-4, 2ep"),
    ("/content/drive/MyDrive/lora_run4", "Run4: 450ex, lr=2e-5, 1ep"),
    ("/content/drive/MyDrive/lora_run5", "Run5: 450ex, lr=1e-4, ga=8"),
]

for run_path, run_label in run_dirs:
    if not os.path.exists(run_path):
        print(f"{run_label:<45} SKIPPED")
        continue
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto", low_cpu_mem_usage=True, trust_remote_code=True,
    )
    ft = PeftModel.from_pretrained(base, run_path)
    ft.eval()
    loss = compute_loss(ft, holdout)
    print(f"{run_label:<45} Loss: {loss:.4f}")
    del ft, base
    torch.cuda.empty_cache()

print("\nLower loss = better fit to ground truth answers.")

Computing losses...



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Baseline                                      Loss: 8.0052


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Run4: 450ex, lr=2e-5, 1ep                     Loss: 3.2637


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Run5: 450ex, lr=1e-4, ga=8                    Loss: 4.1951

Lower loss = better fit to ground truth answers.


# Problem 7: Final Reflection (10 points)

Now we'll take some time to reflect on this homework. Take some time to discuss the following:

1. What concept did you find the most interesting?
2. Which concepts (if any) do you see being useful towards your goal? Why? If there was none, discuss why.
3. Is there a topic that was discussed during lectures up to the release of the assignment that you wished was covered in the homework? Any from the assignment that you wanted there to be touched upon more?

It was really surprising just how much model performance improved after only a small amount of fine-tuning. Intuitively this makes sense, but even with only a few hundred examples the model's loss dropped by almost 75%.

This entire pset is extremely relevant towards my project, since the project involves fine-tuning several copies of Kimi on thousands of training examples with different loss functions for urgent and non-urgent scenarios to see if we can get it to change its behavior in ways we find desirable.

I think that the homework has done a pretty good job of covering the topics from lecture, and there is not really anything which I feel is sorely lacking.